# Hume / Apple Health heart-rate analysis

Converted from `physiology.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Comprehensive Physiological Data Analysis
Heart Rate (HR) and Heart Rate Variability (HRV) Analysis
Aligned with Behavioral Go/No-Go Task Data

Usage:
    python3 physiological_analysis.py                        # auto-discover XML files
    python3 physiological_analysis.py --xml export.xml ...  # explicit XML files
    python3 physiological_analysis.py --subject 5 --output results.txt

This script:
1. Parses ALL Apple HealthKit XML exports whose filename contains 'export' (auto-discovered)
2. Maps physio records to session windows derived from behavioral CSV timestamps
3. Calculates HR, SDNN, and RMSSD metrics per subject/level/set
4. Maps BAPS_s scores from BAPS_Subject{n}_results.csv files (if available)
5. Outputs results in a formatted text file (Physiological_Analysis_Results.txt)

Author: AttentionSpan Research Project


In [ ]:
import xml.etree.ElementTree as ET
import os
import re
import csv
import glob
import math
import statistics
from datetime import datetime, timedelta
from collections import defaultdict
import argparse

# ============================================================================
# CONFIGURATION
# ============================================================================

DEFAULT_XML_PATHS   = None   # auto-discovered at runtime
DEFAULT_OUTPUT_DIR  = "."
DEFAULT_SOURCE_NAME = "Hume"

HRV_LOW_THRESHOLD  = 50
HRV_HIGH_THRESHOLD = 100

HR_RESTING_LOW      = 60
HR_RESTING_HIGH     = 100
HR_CHANGE_THRESHOLD = 5

RECOVERY_WINDOW_BEFORE = 120
RECOVERY_WINDOW_AFTER  = 180

# ============================================================================
# DATA LOADING — XML (supports multiple files, deduplicates by timestamp)
# ============================================================================

def load_physiological_data(xml_paths, source_name="Hume", verbose=True):
    """
    Load and merge physiological data from one or more Apple HealthKit XML exports.
    Records are deduplicated by (type, unix_timestamp) so overlapping exports
    don't produce double-counted samples.

    Returns dict with keys:
        'heart_rate', 'hrv_sdnn', 'spo2', 'body_temp', 'step_count'
    Each value is a list of {'ts': float, 'dt': datetime, 'value': float}.
    """
    if isinstance(xml_paths, str):
        xml_paths = [xml_paths]

    type_mapping = {
        'HKQuantityTypeIdentifierHeartRate':                  'heart_rate',
        'HKQuantityTypeIdentifierHeartRateVariabilitySDNN':   'hrv_sdnn',
        'HKQuantityTypeIdentifierOxygenSaturation':           'spo2',
        'HKQuantityTypeIdentifierBodyTemperature':            'body_temp',
        'HKQuantityTypeIdentifierStepCount':                  'step_count',
    }

    # Use sets to track (type_key, ts) pairs already seen for deduplication
    seen = set()
    data = {k: [] for k in type_mapping.values()}
    total_loaded = 0
    total_dupes  = 0

    for xml_path in xml_paths:
        if not os.path.exists(xml_path):
            if verbose:
                print(f"  [WARNING] XML file not found, skipping: {xml_path}")
            continue

        if verbose:
            print(f"Loading: {xml_path}  (source filter: '{source_name}')")

        tree = ET.parse(xml_path)
        root = tree.getroot()
        file_count = 0

        for record in root.findall(f'.//Record[@sourceName="{source_name}"]'):
            rec_type       = record.get('type')
            start_date_str = record.get('startDate')
            value          = record.get('value')

            if rec_type not in type_mapping or not start_date_str or not value:
                continue

            try:
                dt_str = start_date_str.rsplit(' ', 1)[0]
                dt     = datetime.strptime(dt_str, '%Y-%m-%d %H:%M:%S')
                ts     = dt.timestamp()
                key    = (type_mapping[rec_type], round(ts))

                if key in seen:
                    total_dupes += 1
                    continue

                seen.add(key)
                data[type_mapping[rec_type]].append({'ts': ts, 'dt': dt,
                                                      'value': float(value)})
                file_count  += 1
                total_loaded += 1
            except Exception:
                pass

        if verbose:
            print(f"  → {file_count} new records added from {os.path.basename(xml_path)}")

    # Sort all by timestamp
    for key in data:
        data[key].sort(key=lambda x: x['ts'])

    if verbose:
        print(f"\nTotal records loaded: {total_loaded}  ({total_dupes} duplicates removed)")
        for key, values in data.items():
            if values:
                print(f"  {key}: {len(values)} records")

    return data


# ============================================================================
# BAPS SCORE LOADER  (reads BAPS_Subject{n}_results.csv)
# ============================================================================

def load_baps_scores(base_path=".", verbose=True):
    """
    Scan base_path for BAPS_Subject{n}_results.csv files produced by
    behavioral_attention_analysis.py and extract BAPS_s per level/set.

    Returns dict:
        { subject_num (int): {
            level (int): {
                set_num (int): float   # BAPS_s value
            }
          }
        }
    """
    pattern   = os.path.join(base_path, "BAPS_Subject*_results.csv")
    baps_data = {}

    for fpath in sorted(glob.glob(pattern)):
        m = re.search(r'Subject(\d+)', os.path.basename(fpath))
        if not m:
            continue
        subj = int(m.group(1))
        try:
            with open(fpath, newline='', encoding='utf-8', errors='replace') as fh:
                reader = csv.DictReader(fh)
                subj_data = {}
                for row in reader:
                    try:
                        lvl = int(row.get('Level') or 0)
                        st  = int(row.get('Set')   or 0)
                        val = row.get('BAPS_s', '')
                        if lvl and st and val:
                            subj_data.setdefault(lvl, {})[st] = float(val)
                    except Exception:
                        pass
                if subj_data:
                    baps_data[subj] = subj_data
        except Exception:
            pass

    if verbose:
        if baps_data:
            print(f"\nBAPS_s scores loaded for {len(baps_data)} subjects: "
                  f"{sorted(baps_data.keys())}")
        else:
            print("\nNo BAPS_Subject*_results.csv files found — "
                  "BAPS scores will not appear in output.")

    return baps_data


# ============================================================================
# CSV-BASED SESSION TIME HELPERS
# (reads behavioral .csv files rather than XDF/EEG files)
# ============================================================================

def _parse_csv_timestamps(csv_path):
    """
    Read the 'timestamp' column from a behavioral results CSV and return a
    list of unix timestamps (float).  Handles ISO format with or without
    fractional seconds: '2026-02-03 10:34:28.208' or '2026-02-03 10:34:28'.
    """
    timestamps = []
    try:
        with open(csv_path, 'r', newline='', encoding='utf-8', errors='replace') as fh:
            reader = csv.DictReader(fh)
            for row in reader:
                ts_str = (row.get('timestamp') or '').strip()
                if not ts_str:
                    continue
                # Truncate to seconds precision for strptime
                ts_str = ts_str[:19]
                try:
                    dt = datetime.strptime(ts_str, '%Y-%m-%d %H:%M:%S')
                    timestamps.append(dt.timestamp())
                except Exception:
                    pass
    except Exception:
        pass
    return timestamps


def get_csv_session_times(subject_num, level_num, base_path="."):
    """
    Return {'start', 'end', 'duration_min'} derived from all behavioral CSVs
    for this subject/level, or None if no CSVs are found.
    """
    for subj_dir in [f"Subject{subject_num}", f"subject{subject_num}"]:
        csv_dir = os.path.join(base_path, f"Level{level_num}",
                               "Results_CSV", subj_dir)
        if not os.path.exists(csv_dir):
            continue

        all_ts = []
        for csv_file in glob.glob(os.path.join(csv_dir, "*.csv")):
            all_ts.extend(_parse_csv_timestamps(csv_file))

        if all_ts:
            return {
                'start':        min(all_ts),
                'end':          max(all_ts),
                'duration_min': (max(all_ts) - min(all_ts)) / 60,
            }

    return None


def get_csv_set_times(subject_num, level_num, base_path="."):
    """
    Return {set_num: {'start', 'end', 'duration_min'}} keyed by set (1/2/3),
    derived from the per-set behavioral CSVs, or None if none are found.

    CSV files are matched by the pattern *_set{N}_*.csv (e.g.
    Classic_results_set1_20260203_103620.csv).
    """
    for subj_dir in [f"Subject{subject_num}", f"subject{subject_num}"]:
        csv_dir = os.path.join(base_path, f"Level{level_num}",
                               "Results_CSV", subj_dir)
        if not os.path.exists(csv_dir):
            continue

        set_times = {}
        for set_num in [1, 2, 3]:
            # Try the most specific pattern first, then a broader one
            patterns = [
                os.path.join(csv_dir, f"*_set{set_num}_*.csv"),
                os.path.join(csv_dir, f"*set{set_num}*.csv"),
            ]
            set_files = []
            for p in patterns:
                set_files = glob.glob(p)
                if set_files:
                    break

            set_ts = []
            for csv_file in set_files:
                set_ts.extend(_parse_csv_timestamps(csv_file))

            if set_ts:
                set_times[set_num] = {
                    'start':        min(set_ts),
                    'end':          max(set_ts),
                    'duration_min': (max(set_ts) - min(set_ts)) / 60,
                }

        return set_times if set_times else None

    return None


# Keep old XDF-based helpers as unused fallbacks (not called any more)
def get_eeg_session_times(subject_num, level_num, base_path="."):
    """Legacy XDF-based helper — use get_csv_session_times instead."""
    return get_csv_session_times(subject_num, level_num, base_path)


def get_set_times(subject_num, level_num, base_path="."):
    """Legacy XDF-based helper — use get_csv_set_times instead."""
    return get_csv_set_times(subject_num, level_num, base_path)


# ============================================================================
# ANALYSIS FUNCTIONS
# ============================================================================

def find_matching_records(records, start_ts, end_ts):
    return [r for r in records if start_ts <= r['ts'] <= end_ts]


def calculate_hr_metrics(hr_records):
    if not hr_records:
        return {'count': 0, 'mean': None, 'available': False}

    values = [r['value'] for r in hr_records]
    n = len(values)

    metrics = {
        'count':     n,
        'available': True,
        'mean':      statistics.mean(values),
        'min':       min(values),
        'max':       max(values),
        'range':     max(values) - min(values),
    }

    if n >= 2:
        metrics['std'] = statistics.stdev(values)
        mid = n // 2
        if mid > 0:
            first_half  = statistics.mean(values[:mid])
            second_half = statistics.mean(values[mid:])
            metrics['change'] = second_half - first_half
            if   metrics['change'] >  HR_CHANGE_THRESHOLD: metrics['trend'] = 'increasing'
            elif metrics['change'] < -HR_CHANGE_THRESHOLD: metrics['trend'] = 'decreasing'
            else:                                           metrics['trend'] = 'stable'
        else:
            metrics['change'] = 0
            metrics['trend']  = 'stable'
    else:
        metrics['std']    = 0
        metrics['change'] = 0
        metrics['trend']  = 'stable'

    # RMSSD approximation from sequential HR samples
    # Convert HR → pseudo-RR interval (ms), then compute root-mean-square
    # of successive differences.  NOTE: these are ~2-min sampled readings,
    # not true beat-to-beat intervals, so treat this as a coarse estimate.
    metrics['rmssd'] = calculate_rmssd_from_hr(hr_records)

    return metrics


def calculate_rmssd_from_hr(hr_records):
    """
    Compute RMSSD from sequential HR readings by converting each to a
    pseudo-RR interval (RR = 60 000 / HR_bpm ms) and then taking the
    root-mean-square of successive differences.

    Because Hume samples HR roughly every 2 minutes (not beat-to-beat),
    this is a coarse approximation — useful for relative comparison across
    sessions but not directly comparable to clinical RMSSD values.

    Returns float (ms) or None if fewer than 2 records are available.
    """
    if len(hr_records) < 2:
        return None

    rr = [60_000.0 / r['value'] for r in hr_records if r['value'] > 0]
    if len(rr) < 2:
        return None

    sq_diffs = [(rr[i + 1] - rr[i]) ** 2 for i in range(len(rr) - 1)]
    return math.sqrt(sum(sq_diffs) / len(sq_diffs))


def calculate_hrv_metrics(hrv_records):
    """Calculate SDNN HRV metrics."""
    if not hrv_records:
        return {'count': 0, 'mean': None, 'available': False}

    values = [r['value'] for r in hrv_records]
    n      = len(values)
    mean_sdnn = statistics.mean(values)

    metrics = {
        'count':     n,
        'available': True,
        'mean':      mean_sdnn,
        'min':       min(values),
        'max':       max(values),
    }

    if n >= 2:
        metrics['std'] = statistics.stdev(values)
    else:
        metrics['std'] = 0

    if   mean_sdnn < HRV_LOW_THRESHOLD:  metrics['interpretation'] = 'low (possible stress)'
    elif mean_sdnn > HRV_HIGH_THRESHOLD: metrics['interpretation'] = 'high (excellent)'
    else:                                metrics['interpretation'] = 'normal'

    return metrics


def analyze_recovery(physio_data, stimulus_times,
                     window_before=120, window_after=180):
    if not stimulus_times:
        return {'available': False, 'reason': 'no stimulus times provided'}

    hr_records  = physio_data.get('heart_rate', [])
    hrv_records = physio_data.get('hrv_sdnn', [])

    pre_hr_values  = []
    post_hr_values = []
    pre_hrv_values = []
    post_hrv_values = []

    for stim_ts in stimulus_times:
        pre_hr_values.extend(
            r['value'] for r in hr_records
            if stim_ts - window_before <= r['ts'] < stim_ts)
        post_hr_values.extend(
            r['value'] for r in hr_records
            if stim_ts < r['ts'] <= stim_ts + window_after)
        pre_hrv_values.extend(
            r['value'] for r in hrv_records
            if stim_ts - window_before <= r['ts'] < stim_ts)
        post_hrv_values.extend(
            r['value'] for r in hrv_records
            if stim_ts < r['ts'] <= stim_ts + window_after)

    recovery = {'available': True}

    if pre_hr_values and post_hr_values:
        recovery['hr_pre_mean']   = statistics.mean(pre_hr_values)
        recovery['hr_post_mean']  = statistics.mean(post_hr_values)
        recovery['hr_change']     = recovery['hr_post_mean'] - recovery['hr_pre_mean']
        recovery['hr_pre_count']  = len(pre_hr_values)
        recovery['hr_post_count'] = len(post_hr_values)
        if   recovery['hr_change'] >  5: recovery['hr_pattern'] = 'stress response (HR increased)'
        elif recovery['hr_change'] < -5: recovery['hr_pattern'] = 'recovery (HR decreased)'
        else:                            recovery['hr_pattern'] = 'stable'
    else:
        recovery['hr_available'] = False

    if pre_hrv_values and post_hrv_values:
        recovery['hrv_pre_mean']   = statistics.mean(pre_hrv_values)
        recovery['hrv_post_mean']  = statistics.mean(post_hrv_values)
        recovery['hrv_change']     = recovery['hrv_post_mean'] - recovery['hrv_pre_mean']
        recovery['hrv_pre_count']  = len(pre_hrv_values)
        recovery['hrv_post_count'] = len(post_hrv_values)
        if   recovery['hrv_change'] < -10: recovery['hrv_pattern'] = 'stress response (HRV decreased)'
        elif recovery['hrv_change'] >  10: recovery['hrv_pattern'] = 'recovery (HRV increased)'
        else:                              recovery['hrv_pattern'] = 'stable'
    else:
        recovery['hrv_available'] = False

    return recovery


def compare_sets(set_metrics):
    comparison = {}

    if all(set_metrics.get(s, {}).get('hr', {}).get('available') for s in [1, 2]):
        baseline_hr = set_metrics[1]['hr']['mean']
        set2_hr     = set_metrics[2]['hr']['mean']
        comparison['hr_baseline']       = baseline_hr
        comparison['hr_set2']           = set2_hr
        comparison['hr_set2_change']    = set2_hr - baseline_hr
        comparison['hr_set2_change_pct'] = ((set2_hr - baseline_hr) / baseline_hr) * 100
        if set_metrics.get(3, {}).get('hr', {}).get('available'):
            set3_hr = set_metrics[3]['hr']['mean']
            comparison['hr_set3']        = set3_hr
            comparison['hr_set3_change'] = set3_hr - baseline_hr
            comparison['hr_recovery']    = set2_hr - set3_hr

    if all(set_metrics.get(s, {}).get('hrv', {}).get('available') for s in [1, 2]):
        baseline_hrv = set_metrics[1]['hrv']['mean']
        set2_hrv     = set_metrics[2]['hrv']['mean']
        comparison['hrv_baseline']    = baseline_hrv
        comparison['hrv_set2']        = set2_hrv
        comparison['hrv_set2_change'] = set2_hrv - baseline_hrv
        if set_metrics.get(3, {}).get('hrv', {}).get('available'):
            set3_hrv = set_metrics[3]['hrv']['mean']
            comparison['hrv_set3']        = set3_hrv
            comparison['hrv_set3_change'] = set3_hrv - baseline_hrv

    return comparison


# ============================================================================
# SUBJECT COVERAGE DETECTION
# ============================================================================

def detect_covered_subjects(physio_data, candidate_subjects, base_path=".",
                            verbose=True):
    """
    For each candidate subject, check whether any physiological records fall
    within their EEG session windows.  Prints a coverage summary to the
    terminal and returns a list of subjects that have matching data.
    """
    hr_records = physio_data.get('heart_rate', [])

    covered     = []
    not_covered = []

    print("\n" + "=" * 60)
    print("SUBJECT COVERAGE CHECK")
    print("(based on combined XML physiological data vs behavioral CSV session times)")
    print("=" * 60)

    for subj in sorted(candidate_subjects):
        has_data = False
        levels_found = []
        for level in [1, 2, 3]:
            session = get_csv_session_times(subj, level, base_path)
            if session:
                matching = find_matching_records(
                    hr_records, session['start'], session['end'])
                if matching:
                    has_data = True
                    levels_found.append(level)

        if has_data:
            covered.append(subj)
            print(f"  Subject {subj:>3}:  COVERED  (levels with HR data: "
                  f"{levels_found})")
        else:
            not_covered.append(subj)
            print(f"  Subject {subj:>3}:  no physiological data found")

    print()
    print(f"Covered subjects ({len(covered)}): {covered}")
    print(f"No physio data  ({len(not_covered)}): {not_covered}")
    print("=" * 60)

    return covered


# ============================================================================
# MAIN ANALYSIS FUNCTION
# ============================================================================

def analyze_subject(subject_num, physio_data, base_path=".",
                    stimulus_times=None, baps_scores=None, verbose=True):
    """
    Perform comprehensive physiological analysis for a single subject.
    baps_scores: optional dict {level: {set_num: float}}  (BAPS_s values)
    """
    if verbose:
        print(f"\nAnalyzing Subject {subject_num}...")

    results = {
        'subject': subject_num,
        'levels':  {},
        'summary': {},
        'baps':    baps_scores or {},
    }

    all_hr_means   = []
    all_hrv_means  = []
    all_rmssd_vals = []

    for level in [1, 2, 3]:
        session = get_csv_session_times(subject_num, level, base_path)
        if not session:
            if verbose:
                print(f"  Level {level}: No behavioral CSV data found")
            continue

        level_results = {
            'session_start': datetime.fromtimestamp(session['start']),
            'session_end':   datetime.fromtimestamp(session['end']),
            'duration_min':  session['duration_min'],
            'sets':          {},
        }

        set_times = get_csv_set_times(subject_num, level, base_path)

        if set_times:
            for set_num, times in set_times.items():
                hr_records  = find_matching_records(
                    physio_data['heart_rate'], times['start'], times['end'])
                hrv_records = find_matching_records(
                    physio_data['hrv_sdnn'], times['start'], times['end'])

                set_results = {
                    'start':        datetime.fromtimestamp(times['start']),
                    'end':          datetime.fromtimestamp(times['end']),
                    'duration_min': times['duration_min'],
                    'hr':           calculate_hr_metrics(hr_records),
                    'hrv':          calculate_hrv_metrics(hrv_records),
                    'baps_s':       (baps_scores or {}).get(level, {}).get(set_num),
                }

                if stimulus_times and level in stimulus_times \
                        and set_num in stimulus_times[level]:
                    set_results['recovery'] = analyze_recovery(
                        physio_data, stimulus_times[level][set_num])

                level_results['sets'][set_num] = set_results

                if set_results['hr']['available']:
                    all_hr_means.append(set_results['hr']['mean'])
                    if set_results['hr']['rmssd'] is not None:
                        all_rmssd_vals.append(set_results['hr']['rmssd'])
                if set_results['hrv']['available']:
                    all_hrv_means.append(set_results['hrv']['mean'])

            set_metrics = {s: level_results['sets'].get(s, {}) for s in [1, 2, 3]}
            level_results['comparison'] = compare_sets(set_metrics)

        else:
            hr_records  = find_matching_records(
                physio_data['heart_rate'], session['start'], session['end'])
            hrv_records = find_matching_records(
                physio_data['hrv_sdnn'], session['start'], session['end'])

            level_results['hr']  = calculate_hr_metrics(hr_records)
            level_results['hrv'] = calculate_hrv_metrics(hrv_records)

            if level_results['hr']['available']:
                all_hr_means.append(level_results['hr']['mean'])
                if level_results['hr']['rmssd'] is not None:
                    all_rmssd_vals.append(level_results['hr']['rmssd'])
            if level_results['hrv']['available']:
                all_hrv_means.append(level_results['hrv']['mean'])

        results['levels'][level] = level_results

        if verbose:
            hr_info = level_results.get('hr',
                      level_results.get('sets', {}).get(1, {}).get('hr', {}))
            if hr_info.get('available'):
                print(f"  Level {level}: {hr_info.get('count', 0)} HR samples, "
                      f"mean = {hr_info.get('mean', 0):.1f} bpm")
            else:
                print(f"  Level {level}: Session found but limited physiological data")

    if all_hr_means:
        results['summary']['overall_hr_mean']    = statistics.mean(all_hr_means)
    if all_hrv_means:
        results['summary']['overall_hrv_mean']   = statistics.mean(all_hrv_means)
    if all_rmssd_vals:
        results['summary']['overall_rmssd_mean'] = statistics.mean(all_rmssd_vals)

    return results


def analyze_all_subjects(physio_data, subjects=None, base_path=".",
                         stimulus_times=None, baps_data=None, verbose=True):
    if subjects is None:
        subjects = list(range(2, 39))

    all_results = {}
    for subject in subjects:
        subj_stim  = stimulus_times.get(subject) if stimulus_times else None
        subj_baps  = baps_data.get(subject) if baps_data else None
        results    = analyze_subject(subject, physio_data, base_path,
                                     subj_stim, subj_baps, verbose)
        all_results[subject] = results

    return all_results


# ============================================================================
# OUTPUT FORMATTING
# ============================================================================

def _fmt(val, fmt=".1f", suffix="", na="N/A"):
    if val is None:
        return na
    return f"{val:{fmt}}{suffix}"


def format_results(results, output_path=None):
    lines = []

    def add(text=""):
        lines.append(text)

    add("=" * 80)
    add("PHYSIOLOGICAL DATA ANALYSIS RESULTS")
    add("Heart Rate (HR) and Heart Rate Variability (HRV)")
    add("=" * 80)
    add()

    if 'subject' in results:
        _format_subject_results(results, lines)
    else:
        all_hr    = []
        all_hrv   = []
        all_rmssd = []

        for subj_results in results.values():
            s = subj_results.get('summary', {})
            if s.get('overall_hr_mean'):
                all_hr.append(s['overall_hr_mean'])
            if s.get('overall_hrv_mean'):
                all_hrv.append(s['overall_hrv_mean'])
            if s.get('overall_rmssd_mean'):
                all_rmssd.append(s['overall_rmssd_mean'])

        add("AGGREGATE STATISTICS")
        add("-" * 40)
        if all_hr:
            add(f"Overall Mean HR:           {statistics.mean(all_hr):.1f} bpm")
            add(f"HR Range:                  {min(all_hr):.1f} - {max(all_hr):.1f} bpm")
        if all_hrv:
            add(f"Overall Mean HRV (SDNN):   {statistics.mean(all_hrv):.1f} ms")
            add(f"HRV Range:                 {min(all_hrv):.1f} - {max(all_hrv):.1f} ms")
        if all_rmssd:
            add(f"Overall Mean RMSSD (est.): {statistics.mean(all_rmssd):.1f} ms")
            add(f"RMSSD Range:               {min(all_rmssd):.1f} - {max(all_rmssd):.1f} ms")
        add()

        for subj_num, subj_results in sorted(results.items()):
            add()
            _format_subject_results(subj_results, lines)

    add()
    add("=" * 80)
    add("INTERPRETATION GUIDE")
    add("=" * 80)
    add()
    add("HEART RATE (HR):")
    add("  - Normal resting: 60-100 bpm")
    add("  - During cognitive tasks: typically 5-15 bpm above resting")
    add("  - High variability (SD > 15): stress reactivity or fluctuating arousal")
    add("  - Increasing trend: fatigue or stress accumulation")
    add("  - Decreasing trend: habituation or calming")
    add()
    add("HRV (SDNN):")
    add("  - < 50 ms: Low - possible stress, reduced adaptability")
    add("  - 50-100 ms: Normal - good autonomic function")
    add("  - > 100 ms: High - excellent stress resilience")
    add()
    add("HRV (RMSSD — estimated):")
    add("  - Computed from pseudo-RR intervals derived from ~2-min sampled HR.")
    add("  - Not clinically equivalent to true beat-to-beat RMSSD.")
    add("  - Useful for relative within-subject comparisons across sessions.")
    add("  - Lower values → less parasympathetic variability (more stress).")
    add()
    add("ARS (Attentional Resilience Score):")
    add("  - Baseline (Set 1) = 100 (reference).")
    add("  - Set 2 / Set 3 scores are % of baseline performance under distraction.")
    add("  - > 100: better than baseline (practice effect / heightened arousal).")
    add("  - 85-100: good — minor degradation.")
    add("  - 70-85: moderate impact.")
    add("  - < 70: substantial attentional disruption.")
    add()
    add("=" * 80)
    add("END OF REPORT")
    add("=" * 80)

    output = "\n".join(lines)

    if output_path:
        with open(output_path, 'w') as f:
            f.write(output)
        print(f"\nResults saved to: {output_path}")
    else:
        print(output)

    return output


def _format_subject_results(results, lines):
    def add(text=""):
        lines.append(text)

    add("=" * 60)
    add(f"SUBJECT {results['subject']}")
    add("=" * 60)

    baps_all = results.get('baps', {})

    for level in [1, 2, 3]:
        if level not in results['levels']:
            continue

        level_data = results['levels'][level]
        add()
        add(f"Level {level}")
        add("-" * 40)
        add(f"Session: {level_data['session_start'].strftime('%Y-%m-%d %H:%M')} - "
            f"{level_data['session_end'].strftime('%H:%M')}")
        add(f"Duration: {level_data['duration_min']:.1f} minutes")

        set_names = {1: "Baseline", 2: "Distraction+Respond", 3: "Distraction+Ignore"}

        if 'sets' in level_data:
            for set_num in [1, 2, 3]:
                if set_num not in level_data['sets']:
                    continue

                set_data = level_data['sets'][set_num]
                add()
                add(f"  Set {set_num} ({set_names.get(set_num, '')}):")

                # Heart Rate
                hr = set_data.get('hr', {})
                if hr.get('available'):
                    add(f"    Heart Rate: {hr['count']} samples")
                    add(f"      Mean:          {hr['mean']:.1f} bpm")
                    add(f"      Range:         {hr['min']:.0f} - {hr['max']:.0f} bpm")
                    add(f"      Variability:   {hr['std']:.1f} bpm (SD)")
                    add(f"      Session change:{hr['change']:+.1f} bpm ({hr['trend']})")
                    if hr['rmssd'] is not None:
                        add(f"      RMSSD (est.):  {hr['rmssd']:.1f} ms")
                    else:
                        add(f"      RMSSD (est.):  N/A (< 2 HR samples)")

                # HRV SDNN
                hrv = set_data.get('hrv', {})
                if hrv.get('available'):
                    add(f"    HRV (SDNN): {hrv['count']} samples")
                    add(f"      Mean: {hrv['mean']:.1f} ms ({hrv['interpretation']})")

                # BAPS_s score (from behavioral_attention_analysis.py output)
                baps_s = set_data.get('baps_s')
                if baps_s is not None:
                    add(f"    BAPS_s Score: {baps_s:.1f}")

                # Recovery analysis
                recovery = set_data.get('recovery', {})
                if recovery.get('available'):
                    add(f"    Recovery Analysis:")
                    if recovery.get('hr_pre_mean'):
                        add(f"      HR pre-stim:  {recovery['hr_pre_mean']:.1f} bpm "
                            f"(n={recovery['hr_pre_count']})")
                        add(f"      HR post-stim: {recovery['hr_post_mean']:.1f} bpm "
                            f"(n={recovery['hr_post_count']})")
                        add(f"      HR change:    {recovery['hr_change']:+.1f} bpm — "
                            f"{recovery['hr_pattern']}")
                    if recovery.get('hrv_pre_mean'):
                        add(f"      HRV change:   {recovery['hrv_change']:+.1f} ms — "
                            f"{recovery['hrv_pattern']}")

            # Cross-set comparison
            comp = level_data.get('comparison', {})
            if comp:
                add()
                add("  Cross-Set Comparison:")
                add("    (HR deltas = Set k mean − Set 1 mean, same level; 0 = unchanged vs Set 1.)")
                if 'hr_baseline' in comp:
                    add(f"    HR  Baseline→Set2: {comp['hr_baseline']:.1f} → "
                        f"{comp['hr_set2']:.1f} bpm ({comp['hr_set2_change']:+.1f})")
                    if 'hr_set3' in comp:
                        add(f"    HR  Baseline→Set3: {comp['hr_baseline']:.1f} → "
                            f"{comp['hr_set3']:.1f} bpm ({comp['hr_set3_change']:+.1f})")
                        if comp.get('hr_recovery'):
                            add(f"    HR  Recovery Set2→Set3: {comp['hr_recovery']:+.1f} bpm")
                if 'hrv_baseline' in comp:
                    add(f"    HRV Baseline→Set2: {comp['hrv_baseline']:.1f} → "
                        f"{comp['hrv_set2']:.1f} ms ({comp['hrv_set2_change']:+.1f})")
                    if 'hrv_set3' in comp:
                        add(f"    HRV Baseline→Set3: {comp['hrv_baseline']:.1f} → "
                            f"{comp['hrv_set3']:.1f} ms ({comp['hrv_set3_change']:+.1f})")

            # BAPS_s summary for this level
            level_baps = baps_all.get(level)
            if level_baps:
                add()
                add("  BAPS_s Summary:")
                for sn in [1, 2, 3]:
                    bval = level_baps.get(sn)
                    if bval is not None:
                        add(f"    Set {sn}: BAPS_s = {bval:>6.1f}")

        else:
            hr = level_data.get('hr', {})
            if hr.get('available'):
                add(f"  Heart Rate: {hr['count']} samples")
                add(f"    Mean:          {hr['mean']:.1f} bpm")
                add(f"    Range:         {hr['min']:.0f} - {hr['max']:.0f} bpm")
                add(f"    Variability:   {hr['std']:.1f} bpm (SD)")
                add(f"    Session change:{hr['change']:+.1f} bpm ({hr['trend']})")
                if hr['rmssd'] is not None:
                    add(f"    RMSSD (est.):  {hr['rmssd']:.1f} ms")

            hrv = level_data.get('hrv', {})
            if hrv.get('available'):
                add(f"  HRV (SDNN): {hrv['count']} samples")
                add(f"    Mean: {hrv['mean']:.1f} ms ({hrv['interpretation']})")

            # BAPS_s summary (level-wide fallback — no per-set breakdown)
            level_baps = baps_all.get(level)
            if level_baps:
                add()
                add("  BAPS_s Summary:")
                for sn in [1, 2, 3]:
                    bval = level_baps.get(sn)
                    if bval is not None:
                        add(f"    Set {sn}: BAPS_s = {bval:>6.1f}")

    summary = results.get('summary', {})
    if summary:
        add()
        add("Overall Summary:")
        if summary.get('overall_hr_mean'):
            add(f"  Mean HR across sessions:    {summary['overall_hr_mean']:.1f} bpm")
        if summary.get('overall_hrv_mean'):
            add(f"  Mean SDNN across sessions:  {summary['overall_hrv_mean']:.1f} ms")
        if summary.get('overall_rmssd_mean'):
            add(f"  Mean RMSSD across sessions: {summary['overall_rmssd_mean']:.1f} ms (est.)")


# ============================================================================
# STIMULUS TIME PARSING
# ============================================================================

def parse_stimulus_times(stimulus_input):
    if not stimulus_input or not stimulus_input.strip():
        return []

    times      = [t.strip() for t in stimulus_input.split(',')]
    unix_times = []

    for t in times:
        if not t:
            continue
        if 'am' in t.lower() or 'pm' in t.lower():
            try:
                t_clean = t.lower().replace(' ', '')
                if 'pm' in t_clean:
                    t_clean = t_clean.replace('pm', '')
                    parts = t_clean.split(':')
                    hour  = int(parts[0])
                    if hour != 12:
                        hour += 12
                    minute = int(parts[1]) if len(parts) > 1 else 0
                else:
                    t_clean = t_clean.replace('am', '')
                    parts   = t_clean.split(':')
                    hour    = int(parts[0])
                    if hour == 12:
                        hour = 0
                    minute = int(parts[1]) if len(parts) > 1 else 0
                unix_times.append(hour * 3600 + minute * 60)
            except Exception:
                pass
        else:
            try:
                val = float(t.replace('s', ''))
                unix_times.append(val)
            except Exception:
                pass

    return unix_times


def interactive_stimulus_input():
    print("\n" + "=" * 60)
    print("STIMULUS TIME INPUT")
    print("=" * 60)
    print("\nEnter stimulus times for recovery analysis.")
    print("Supported formats: 1:30pm / 0, 60, 120 / unix timestamps")
    print("Press Enter to skip.\n")

    stimulus_times = {}
    for level in [1, 2, 3]:
        stimulus_times[level] = {}
        print(f"\n--- Level {level} ---")
        for set_num in [2, 3]:
            set_name   = "Distraction+Respond" if set_num == 2 else "Distraction+Ignore"
            user_input = input(f"  Set {set_num} ({set_name}) stimulus times: ").strip()
            if user_input:
                times = parse_stimulus_times(user_input)
                if times:
                    stimulus_times[level][set_num] = times
                    print(f"    Added {len(times)} stimulus times")

    return stimulus_times


# ============================================================================
# MAIN ENTRY POINT
# ============================================================================


In [ ]:
def main():
    parser = argparse.ArgumentParser(
        description="Physiological Data Analysis for Attention Tasks",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  python3 physiological_analysis.py
  python3 physiological_analysis.py --xml export.xml export_2.xml
  python3 physiological_analysis.py --subject 5 --output results.txt
  python3 physiological_analysis.py --interactive-stim
        """)

    parser.add_argument('--xml', type=str, nargs='+', default=None,
                        help="Path(s) to HealthKit XML export(s). "
                             "If omitted, all *export*.xml files in --base-path "
                             "are discovered automatically.")
    parser.add_argument('--source', type=str, default=DEFAULT_SOURCE_NAME,
                        help=f"Filter by source name (default: {DEFAULT_SOURCE_NAME})")
    parser.add_argument('--subject', type=int, nargs='+',
                        help="Subject number(s) to analyze (default: 2-38)")
    parser.add_argument('--output', type=str, default=None,
                        help="Output file path (default: auto-named txt file)")
    parser.add_argument('--base-path', type=str, default=".",
                        help="Base path for behavioral CSV and BAPS results directories")
    parser.add_argument('--interactive-stim', action='store_true',
                        help="Interactively input stimulus times for recovery analysis")
    parser.add_argument('--quiet', action='store_true',
                        help="Suppress progress messages")

    args    = parser.parse_args()
    verbose = not args.quiet

    # ── Resolve XML paths: explicit or auto-discover *export*.xml ────────────
    if args.xml:
        xml_paths = args.xml
    else:
        search_dir = args.base_path
        xml_paths  = sorted(glob.glob(os.path.join(search_dir, "*export*.xml")))
        if not xml_paths:
            # Also check the current working directory
            xml_paths = sorted(glob.glob("*export*.xml"))
        if xml_paths:
            if verbose:
                print(f"Auto-discovered {len(xml_paths)} XML file(s):")
                for p in xml_paths:
                    print(f"  {p}")
        else:
            print("ERROR: No XML files containing 'export' found.  "
                  "Use --xml to specify paths explicitly.")
            return 1

    # ── Load & merge physiological data from all XML files ──────────────────
    try:
        physio_data = load_physiological_data(xml_paths, args.source, verbose)
    except Exception as e:
        print(f"Error loading XML data: {e}")
        return 1

    if not any(physio_data.values()):
        print(f"Error: No data found for source '{args.source}' in {args.xml}")
        return 1

    # ── Load BAPS_s scores from BAPS_Subject{n}_results.csv files ───────────
    baps_data = load_baps_scores(args.base_path, verbose)

    # ── Determine candidate subject list ────────────────────────────────────
    if args.subject:
        subjects = args.subject
    else:
        # Union of 2-38 and any subjects that have BAPS CSV files
        subjects = sorted(set(list(range(2, 39))) | set(baps_data.keys()))

    # ── Show coverage in terminal ────────────────────────────────────────────
    covered = detect_covered_subjects(physio_data, subjects,
                                      args.base_path, verbose)

    # ── Collect stimulus times ───────────────────────────────────────────────
    stimulus_times = None
    if args.interactive_stim:
        stimulus_times = interactive_stimulus_input()

    # ── Run analysis (only covered subjects unless manually specified) ───────
    subjects_to_analyse = args.subject if args.subject else covered
    if verbose:
        print(f"\nRunning physiological analysis for {len(subjects_to_analyse)} "
              f"subjects: {subjects_to_analyse}")

    results = analyze_all_subjects(
        physio_data,
        subjects       = subjects_to_analyse,
        base_path      = args.base_path,
        stimulus_times = stimulus_times,
        baps_data      = baps_data,
        verbose        = verbose,
    )

    # ── Output ───────────────────────────────────────────────────────────────
    output_path = args.output
    if not output_path:
        if len(subjects_to_analyse) == 1:
            output_path = f"Physiological_Subject{subjects_to_analyse[0]}.txt"
        else:
            output_path = "Physiological_Analysis_Results.txt"

    format_results(results, output_path)
    return 0


In [ ]:
# Run this cell to execute the script
exit(main())
